In [50]:
import os
import re
import pandas as pd
from tqdm import tqdm

# Paths
DATA_PATH = '/data/gusev/USERS/jpconnor/data/'
EMBED_PROJ_PATH = os.path.join(DATA_PATH, 'clinical_text_embedding_project/')
NEPC_PROJ_PATH = os.path.join(DATA_PATH, 'CAIA/COMPASS/')

PROC_PATH = os.path.join(EMBED_PROJ_PATH, 'batched_datasets/processed_datasets/')
TEXT_PATH = os.path.join(EMBED_PROJ_PATH, 'batched_datasets/batched_text/')

PROFILE_PATH = '/data/gusev/PROFILE/CLINICAL/'
INTAE_DATA_PATH = os.path.join(PROFILE_PATH, 'robust_VTE_pred_project_2025_03_cohort/data/')
ONCDRS_PATH = os.path.join(PROFILE_PATH, 'OncDRS/ALL_2025_03/')

SURV_PATH = os.path.join(EMBED_PROJ_PATH, 'time-to-event_analysis/')

os.listdir(NEPC_PROJ_PATH)

lab_code_mapping = pd.read_csv('/data/gusev/USERS/jpconnor/code/CAIA/COMPASS/PROFILE/OMOP_to_DFCI_lab_ids.csv')

In [51]:
# DFCI_MRN, DATE, Test DESCR, Numeric_result, text_result, unit

In [73]:
health_df = pd.read_csv(os.path.join(NEPC_PROJ_PATH, 'prostate_health_history_data.csv'))
labs_df = pd.read_csv(os.path.join(NEPC_PROJ_PATH, 'prostate_labs_data.csv')) 

vital_signs_df = health_df.loc[health_df['CODE_TYPE'] == 'Vital Signs', 
                               ['DFCI_MRN', 'START_DT', 'HEALTH_HISTORY_TYPE', 'RESULTS', 'UNITS_CD']]

In [74]:
labs_df_col_sub = labs_df[['DFCI_MRN', 'SPECIMEN_COLLECT_DT', 'TEST_TYPE_CD', 'TEST_TYPE_DESCR', 'NUMERIC_RESULT', 'RESULT_UOM_NM']]

def generate_new_test_names(code, descr):
    if str(code).lower() == 'nan':
        return descr
    elif code == descr:
        return code
    else:
        return f'{code} ({descr})'


labs_df_col_sub['TEST_NAME'] = labs_df_col_sub.apply(lambda row : generate_new_test_names(row['TEST_TYPE_CD'], row['TEST_TYPE_DESCR']), axis=1)
labs_df_col_sub = labs_df_col_sub.drop(columns=['TEST_TYPE_CD', 'TEST_TYPE_DESCR'])

vital_signs_df = (vital_signs_df
                  .rename(columns={'START_DT' : 'DATE', 'RESULTS' : 'LAB_VALUE', 
                                   'HEALTH_HISTORY_TYPE' : 'LAB_NAME', 'UNITS_CD' : 'LAB_UNIT'})
                  [['DFCI_MRN', 'DATE', 'LAB_NAME', 'LAB_UNIT', 'LAB_VALUE']])

labs_df_col_sub = (labs_df_col_sub
                   .rename(columns={'SPECIMEN_COLLECT_DT' : 'DATE', 'NUMERIC_RESULT' : 'LAB_VALUE', 
                                    'RESULT_UOM_NM' : 'LAB_UNIT', 'TEST_NAME' : 'LAB_NAME'})
                   [['DFCI_MRN', 'DATE', 'LAB_NAME', 'LAB_UNIT', 'LAB_VALUE']])

/tmp/ipykernel_312912/531633621.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labs_df_col_sub['TEST_NAME'] = labs_df_col_sub.apply(lambda row : generate_new_test_names(row['TEST_TYPE_CD'], row['TEST_TYPE_DESCR']), axis=1)


In [80]:
complete_longitudinal_data = pd.concat([vital_signs_df, labs_df_col_sub])

In [81]:
complete_longitudinal_data

,DFCI_MRN,DATE,LAB_NAME,LAB_UNIT,LAB_VALUE
0,648898,2020-02-10 14:40:00,Height,inch,67.99200
1,746809,2023-10-23 13:55:00,BMI,NaN,28.89000
2,826699,2018-10-11 09:00:00,Weight,pound,146.16500
3,639190,2017-08-04 12:37:00,Systolic-Epic,millimeter of mercury,126.00000
4,782904,2021-01-19 10:55:00,Body Surface Area (BSA),NaN,2.01000
...,...,...,...,...,...
2438506,433630,2018-07-06 13:13:00,ABASOP (ABSOLUTE BASOS),K/uL,0.04
2438507,1177545,2025-02-21 08:04:00,SODIUM,mmol/L,143.0
2438508,601116,2015-11-05 12:13:00,EOSP (EOS),%,1.4
2438509,671292,2017-04-07 06:50:00,RDW,%,13.8


In [82]:
code_descriptions = complete_longitudinal_data[['LAB_NAME', 'LAB_UNIT']].value_counts().reset_index()
code_descriptions.to_csv('/data/gusev/USERS/jpconnor/code/CAIA/COMPASS/PROFILE/unique_lab_ids_w_units.csv', index=False)

In [83]:
code_descriptions

,LAB_NAME,LAB_UNIT,count
0,Systolic-Epic,millimeter of mercury,79830
1,Blood Pressure-Epic,millimeter of mercury,79830
2,Diastolic-Epic,millimeter of mercury,79830
3,Pulse,beats/minute,78695
4,Weight,pound,75122
...,...,...,...
1676,SGPT (ALT | CDR NSMC LAB LRR),IU/L,1
1677,SGPGM (MAG-SGPG IGM),titer,1
1678,CHOL (CHOLESTEROL | CDR EHS LAB LRR),mg/dL,1
1679,CHOL (CHOLESTEROL | CDR CRMA LAB LLR),mg/dL,1
